# The warehouse against its sources

Step 5 built the warehouse: ODS from the raw files, DWD with every version and its `known_at`, DWS through `as_of`, ADS with labels and outcomes. This notebook asks whether what was built is true to the sources, from two sides. The reasons behind each rule are in [notebook 04](04_ods_to_dwd.ipynb); here the built models are checked, read from dbt's datasets in dev.

| Question | Where |
| --- | --- |
| Is every version's `known_at` consistent with the fetches that brought it, for every source? | Section 1 |
| Does one known day, traced by hand from the raw files, come out right in every layer? | Section 2 |
| How exact can each decision time be: which of its inputs were captured, and which rule-dated? | Section 3 |

## 0. Setup

**Step.** `sql()` runs a query against dbt's DWD dataset in dev and prints what it scanned, as in notebooks 04 and 05. `ODS` is the parsers' dataset, read here only for its manifest, `raw_files`: one row per raw file and ODS table, with the fetch time of each file (`fetched_at`) and how it was fetched (`load_type`: `daily`, `initial` or `recapture`).

In [1]:
import os
from pathlib import Path

import pandas as pd
from google.cloud import bigquery

pd.set_option("display.width", 160, "display.max_columns", 30, "display.max_colwidth", 60)


def setting(name: str) -> str:
    if value := os.environ.get(name):
        return value
    lines = Path("../.env").read_text().splitlines()
    env = dict(line.split("=", 1) for line in lines if "=" in line and not line.startswith("#"))
    return env[name]


PROJECT = setting("JTF_BIGQUERY_PROJECT")
DWD = f"{PROJECT}.{setting('JTF_DBT_DATASET_PREFIX')}_dwd"
ODS = f"{PROJECT}.ods"
bq = bigquery.Client(project=PROJECT)


def sql(query: str) -> pd.DataFrame:
    """Run a query against dbt's DWD dataset in dev; print what it scanned and how long it took."""
    config = bigquery.QueryJobConfig(default_dataset=DWD, use_query_cache=False)
    job = bq.query(query, job_config=config)
    rows = job.to_dataframe(create_bqstorage_client=False)
    seconds = (job.ended - job.started).total_seconds()
    print(f"[{(job.total_bytes_processed or 0) / 1e6:.1f} MB scanned, {seconds:.0f} s in BigQuery]")
    return rows


print(f"DWD dataset: {DWD} | ODS: {ODS}")

DWD dataset: jepx-tokyo-forecast-dev.dev_dwd | ODS: jepx-tokyo-forecast-dev.ods


## 1. `known_at` per source

What "right" means differs by source, because sources reach us differently ([notebook 04](04_ods_to_dwd.ipynb), section 1). A version is dated one of two ways, its `basis`:

- **`rule`**: a value we hold only from history, fetched after the fact, is dated by the source's history rule: when the source published it, as notebook 04 worked out per source (sections 2, 3.1, 3.2 and 6)
- **`captured`**: a value first seen by a fetch is known at that fetch, never earlier

The plan names three checks to run on real data after every load (Testing, Warehouse tests): `rule` versions sit before their source's capture start, `captured` versions carry the fetch time of a file holding the value, and no `known_at` lies after the newest fetch. Before turning them into tests for every source, this section asks what each source has been fetched as, and whether the checks hold on the models as built.

### 1.1 How each source reached us

**Step.** Per source in the manifest, how many files came by each kind of fetch, and when the first and last came. A source's *capture start* is its first `daily` fetch (notebook 04, section 1): before it, history is all we have, and the rule dates it.

In [2]:
fetches = sql(f"""
SELECT source, load_type, COUNT(DISTINCT path) AS files,
  MIN(fetched_at) AS first_fetch, MAX(fetched_at) AS last_fetch
FROM `{ODS}.raw_files`
WHERE source IN ('tepco_actuals', 'meti_fit_capacity', 'openmeteo_previous_runs',
                 'openmeteo_single_runs', 'openmeteo_live')
GROUP BY source, load_type
ORDER BY source, load_type
""")
fetches

[0.2 MB scanned, 0 s in BigQuery]


,source,load_type,files,first_fetch,last_fetch
0,meti_fit_capacity,daily,3,2026-11-16 23:45:19.719609+00:00,2026-11-17 23:45:18.096105+00:00
1,meti_fit_capacity,initial,119,2026-11-16 00:40:24.962795+00:00,2026-11-16 00:45:32.972052+00:00
2,openmeteo_live,daily,5,2026-11-15 02:36:27.058501+00:00,2026-11-17 23:45:18.697832+00:00
3,openmeteo_previous_runs,initial,36,2026-11-15 04:52:42.808649+00:00,2026-11-16 12:29:54.176374+00:00
4,openmeteo_single_runs,initial,1362,2026-11-15 05:01:46.615911+00:00,2026-11-16 12:46:48.318503+00:00
5,tepco_actuals,daily,12,2026-11-15 02:04:16.643393+00:00,2026-11-17 23:45:20.405456+00:00
6,tepco_actuals,initial,41,2026-11-15 04:52:40.111927+00:00,2026-11-15 04:54:24.243215+00:00
7,tepco_actuals,recapture,1,2026-11-15 05:02:48.862189+00:00,2026-11-15 05:02:48.862189+00:00


**What it shows.** **Observed:**

- **The actuals and capacity have a daily capture, so a capture start.** The actuals' first daily fetch came at 11:04 JST on 15 November, before their history was loaded at 13:52: the history files fetched after it are known at their fetch, the late backfill of [notebook 04](04_ods_to_dwd.ipynb), section 2.2. Capacity's first daily fetch came at 08:45 on 17 November, after its 119 history files the morning before
- **Previous Runs and Single Runs came only by `initial` fetches**, 36 and 1,362 files on 15 and 16 November. The capture jobs never fetch them (**Source**: [ingest README](../jtf/src/jtf/ingest/README.md): *"history-only sources (`openmeteo_previous_runs`, `openmeteo_single_runs`) have no daily schedule"*), so they have no capture start
- **The live weather came only by daily fetches**, five files since a test run on 15 November

**Learned → design:** "rule versions before capture start" has nothing to compare against for the weather history, the source with the most rule-dated versions. The rule needs a check that every source has: whatever a rule says, it can't date a value later than the fetch that first brought it, since nothing is published after we fetched it. That is check D below.

### 1.2 Five checks over every versioned model

**Step.** Every DWD model others read through `as_of` carries `known_at` and `basis` (the rules' layer test makes sure of the first). Each model is checked whole, against the manifest, by five counts of versions that break a check:

| Check | Counts the versions where |
| --- | --- |
| **A**: captured, never early | `basis = 'captured'` and `known_at` lies before the fetch that held the value |
| **B**: captured, exactly | `basis = 'captured'` and `known_at` differs from that fetch |
| **C**: rule, before capture start | `basis = 'rule'` and `known_at` lies at or after the source's capture start, where it has one |
| **D**: rule, never after its fetch | `basis = 'rule'` and `known_at` lies after the fetch that first held the value |
| **E**: nothing after the newest fetch | `known_at` lies after the newest fetch of the model's sources |

The fetch that held a value is each model's own column: `first_seen` for the actuals, the capacity figures and the weather history, `first_held` for the capacity releases (when both of a release's files were first held), `captured_at` for the live weather. `dwd_weather_slots` has none: it is made from the hours of the three weather models, so only E applies to it. A check that doesn't apply shows as empty.

In [3]:
MODELS = {
    # model: (its sources, the column with the fetch that held a value, its capture start's source)
    "dwd_tepco_actuals": (["tepco_actuals"], "first_seen", "tepco_actuals"),
    "dwd_capacity_releases": (["meti_fit_capacity"], "first_held", "meti_fit_capacity"),
    "dwd_capacity_area": (["meti_fit_capacity"], "first_seen", "meti_fit_capacity"),
    "dwd_weather_previous_runs": (["openmeteo_previous_runs"], "first_seen", None),
    "dwd_weather_single_runs": (["openmeteo_single_runs"], "first_seen", None),
    "dwd_weather_live": (["openmeteo_live"], "captured_at", "openmeteo_live"),
    "dwd_weather_slots": (
        ["openmeteo_previous_runs", "openmeteo_single_runs", "openmeteo_live"],
        None,
        None,
    ),
}


def counts(model: str, sources: list[str], fetch: str | None, start: str | None) -> str:
    def count(condition, applies=True):
        return f"COUNTIF({condition})" if applies else "CAST(NULL AS INT64)"

    listed = "', '".join(sources)
    capture_start = f"(SELECT capture_start FROM starts WHERE source = '{start}')"
    newest = f"(SELECT MAX(newest) FROM starts WHERE source IN ('{listed}'))"
    return f"""
SELECT '{model}' AS model, COUNT(*) AS versions, COUNTIF(basis = 'captured') AS captured,
  {count(f"basis = 'captured' AND known_at < {fetch}", bool(fetch))} AS a_captured_early,
  {count(f"basis = 'captured' AND known_at != {fetch}", bool(fetch))} AS b_captured_not_exact,
  {count(f"basis = 'rule' AND known_at >= {capture_start}", bool(start))} AS c_rule_after_start,
  {count(f"basis = 'rule' AND known_at > {fetch}", bool(fetch))} AS d_rule_after_fetch,
  {count(f"known_at > {newest}")} AS e_after_newest
FROM `{DWD}.{model}`"""


checks = sql(
    f"""WITH starts AS (
  SELECT source, MIN(IF(load_type = 'daily', fetched_at, NULL)) AS capture_start,
    MAX(fetched_at) AS newest
  FROM `{ODS}.raw_files` GROUP BY source
)"""
    + "\nUNION ALL".join(counts(model, *spec) for model, spec in MODELS.items())
)
checks.set_index("model")

[168.6 MB scanned, 0 s in BigQuery]


,versions,captured,a_captured_early,b_captured_not_exact,c_rule_after_start,d_rule_after_fetch,e_after_newest
model,,,,,,,
dwd_tepco_actuals,147075,423,0,0,0,0,0
dwd_capacity_releases,33,0,0,0,0,0,0
dwd_capacity_area,297,0,0,0,0,0,0
dwd_weather_previous_runs,2019029,104,0,0,<NA>,0,0
dwd_weather_single_runs,1274418,0,0,0,<NA>,0,0
dwd_weather_live,9630,9630,0,0,0,0,0
dwd_weather_slots,6606232,19546,<NA>,<NA>,<NA>,<NA>,0


**What it shows.** **Observed**, over 10.1 M versions in the seven models:

- **No version breaks a check that applies to it**
- **Every captured version is known exactly at its fetch** (check B): the actuals' 423, Previous Runs' 104 corrections, all 9,630 live versions
- **Three models have no captured version yet:** the capacity releases and their figures, since every release so far is dated by the rule (the first captured one is due by mid-December), and Single Runs, where no run has changed between fetches. On them, A and B have nothing to check

**Learned → design:** A, C, D and E become data tests after every load. B holds today but is a rule only where a model makes it one: the actuals and the capacity releases date a captured version at its fetch, and their tests already say so. The weather history dates a correction at its fetch or at the version it replaces, whichever is later (**Source**: [weather.sql](../warehouse/dbt/macros/weather.sql), `weather_versions`: *"A later version is known at the fetch that first held it, never before the version it replaces"*). A correction fetched before the previous version's rule time would be known after its fetch, by design. So for the weather the test is A: never before its fetch.

## 2. One known day by hand: target day 17 November

**Step.** A forecast for D = 17 November is made at T = 09:00 JST on 16 November. It may use what was captured by then; its truth arrives with the actuals captured on the 18th. Each input is read here from the raw file the source sent, through `jtf`'s storage, and parsed by hand in pandas, not with `jtf`'s parsers. It is then compared with every layer it passes through:

| Input | Raw file | Section |
| --- | --- | --- |
| The truth: D's actuals | TEPCO's monthly file as captured at 08:45 on the 18th | 2.1 |
| What T knew of D−2, the persistence feature | The same file as captured at 08:45 on the 16th, before T | 2.2 |
| The weather at T | Open-Meteo's live forecast captured at 08:45 on the 16th | 2.3 |
| Capacity at T, and clear sky | METI's releases known at T, through the models checked against notebook 03 | 2.4, 2.5 |

The day sits a day after capture start, so its inputs come from captures, the way a live forecast's will.

In [4]:
import json
from datetime import timedelta
from io import StringIO

import numpy as np

from jtf.ingest.sources import WEATHER_POINTS
from jtf.storage import open_store

RAW = open_store(setting("JTF_RAW_URL"))  # the prod raw bucket, read-only
PREFIX = setting("JTF_DBT_DATASET_PREFIX")
DWS, ADS, DIM = (f"{PROJECT}.{PREFIX}_{layer}" for layer in ("dws", "ads", "dim"))
T = pd.Timestamp("2026-11-16 09:00", tz="Asia/Tokyo")
D = pd.Timestamp("2026-11-17").date()
AT_T = f"TIMESTAMP('{T:%Y-%m-%d %H:%M:%S}', 'Asia/Tokyo')"  # T in SQL
TRUTH = "tepco_actuals/2026-11-18/20261117T234520Z_eria_jukyu_202611_03.csv"
KNOWN = "tepco_actuals/2026-11-16/20261115T234520Z_eria_jukyu_202611_03.csv"
WEATHER = "openmeteo_live/2026-11-16/20261115T234518Z_forecast.json"


def fetched(path: str) -> pd.Timestamp:
    """When the capture fetched a raw file, from the sidecar stored beside it."""
    meta = json.loads(RAW.get(path + ".meta.json"))
    return pd.Timestamp(meta["fetched_at"]).tz_convert("Asia/Tokyo")


def tepco_by_hand(path: str) -> tuple[str, pd.DataFrame]:
    """A TEPCO monthly actuals file, read without jtf's parser: UTF-8 or CP932, the unit on its
    first line, a row per slot labelled by its start (notebook 05, section 4)."""
    raw = RAW.get(path)
    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        text = raw.decode("cp932")
    unit = text.splitlines()[0].split(",")[0]
    rows = pd.read_csv(StringIO(text), skiprows=1)
    start = pd.to_datetime(rows["DATE"] + " " + rows["TIME"], format="%Y/%m/%d %H:%M")
    frame = pd.DataFrame(
        {
            "slot_start": start.dt.tz_localize("Asia/Tokyo"),
            "solar": rows["太陽光発電実績"],  # solar output, actual
            "curtailed": rows["太陽光出力制御量"],  # solar output curtailed
        }
    )
    return unit, frame


for path in (KNOWN, WEATHER, TRUTH):
    print(f"fetched {fetched(path):%d %b %H:%M:%S} JST  {path}")
print(f"T = {T:%d %b %H:%M} JST, D = {D}")

fetched 16 Nov 08:45:20 JST  tepco_actuals/2026-11-16/20261115T234520Z_eria_jukyu_202611_03.csv
fetched 16 Nov 08:45:18 JST  openmeteo_live/2026-11-16/20261115T234518Z_forecast.json
fetched 18 Nov 08:45:20 JST  tepco_actuals/2026-11-18/20261117T234520Z_eria_jukyu_202611_03.csv
T = 16 Nov 09:00 JST, D = 2026-11-17


**What it shows.** The file known at T was fetched at 08:45:20 on the 16th, 15 minutes before T, and the weather two seconds before it; the truth came two days later.

### 2.1 The truth: D's actuals, from the raw file to the label

**Step.** TEPCO's file as captured on the 18th holds every slot of the 17th. Each slot's solar output and curtailment, read by hand, is compared with the ODS row the parser made from the same file, the newest version in `dwd_tepco_actuals`, and `ads_label_solar`, whose label is delivered + curtailed (what the weather allowed). Every slot of the day is compared; five are shown.

In [5]:
unit, truth = tepco_by_hand(TRUTH)
by_hand = truth[truth["slot_start"].dt.date == D]

layers = sql(f"""
WITH ods AS (
  SELECT TIMESTAMP(PARSE_DATETIME('%Y/%m/%d %H:%M', CONCAT(date, ' ', time)), 'Asia/Tokyo')
      AS slot_start,
    solar, solar_curtailed
  FROM `{ODS}.tepco_actuals`
  WHERE source_file = '{TRUTH}'
  QUALIFY ingested_at = MAX(ingested_at) OVER ()
),
newest AS (
  SELECT slot_start, measure, value_mw, basis, known_at
  FROM dwd_tepco_actuals
  WHERE known_until IS NULL AND DATE(slot_start, 'Asia/Tokyo') = '{D}'
)
SELECT o.slot_start, o.solar AS ods_solar, o.solar_curtailed AS ods_curtailed,
  s.value_mw AS dwd_solar, c.value_mw AS dwd_curtailed, s.basis, s.known_at,
  l.delivered_mw, l.curtailed_mw, l.label_mw
FROM ods AS o
LEFT JOIN newest AS s ON s.slot_start = o.slot_start AND s.measure = 'solar'
LEFT JOIN newest AS c ON c.slot_start = o.slot_start AND c.measure = 'solar_curtailed'
LEFT JOIN `{ADS}.ads_label_solar` AS l ON l.slot_start = o.slot_start
WHERE DATE(o.slot_start, 'Asia/Tokyo') = '{D}'
""")
layers["slot_start"] = layers["slot_start"].dt.tz_convert("Asia/Tokyo")
layers["known_at"] = layers["known_at"].dt.tz_convert("Asia/Tokyo")
day = by_hand.merge(layers, on="slot_start", how="outer").sort_values("slot_start")

links = {
    "raw = ODS": (day.solar == day.ods_solar) & (day.curtailed == day.ods_curtailed),
    "ODS = DWD": (day.ods_solar == day.dwd_solar) & (day.ods_curtailed == day.dwd_curtailed),
    "DWD = label's parts": (day.dwd_solar == day.delivered_mw)
    & (day.dwd_curtailed == day.curtailed_mw),
    "label = delivered + curtailed": day.label_mw == day.delivered_mw + day.curtailed_mw,
}
print(f"unit: {unit} | slots: {len(by_hand)} by hand, {len(layers)} in the layers")
print({link: int((~ok).sum()) for link, ok in links.items()}, "slots differ")
print("D's newest versions, by basis and when known:")
print(
    day.groupby(["basis", "known_at"])
    .slot_start.agg(["count", "min", "max"])
    .map(lambda v: v.strftime("%H:%M") if hasattr(v, "strftime") else v)
    .to_string()
)
day[day.slot_start.dt.strftime("%H:%M").isin(["06:30", "09:00", "12:00", "15:00", "16:30"])]

[11.4 MB scanned, 0 s in BigQuery]
unit: 単位[MW平均] | slots: 48 by hand, 48 in the layers
{'raw = ODS': 0, 'ODS = DWD': 0, "DWD = label's parts": 0, 'label = delivered + curtailed': 0} slots differ
D's newest versions, by basis and when known:
                                           count    min    max
basis    known_at                                             
captured 2026-11-17 08:45:21.013205+09:00     17  00:00  08:00
         2026-11-18 08:45:20.405456+09:00     31  08:30  23:30


,slot_start,solar,curtailed,ods_solar,ods_curtailed,dwd_solar,dwd_curtailed,basis,known_at,delivered_mw,curtailed_mw,label_mw
13,2026-11-17 06:30:00+09:00,90,0,90,0,90.0,0.0,captured,2026-11-17 08:45:21.013205+09:00,90.0,0.0,90.0
18,2026-11-17 09:00:00+09:00,766,0,766,0,766.0,0.0,captured,2026-11-18 08:45:20.405456+09:00,766.0,0.0,766.0
24,2026-11-17 12:00:00+09:00,1487,0,1487,0,1487.0,0.0,captured,2026-11-18 08:45:20.405456+09:00,1487.0,0.0,1487.0
30,2026-11-17 15:00:00+09:00,310,0,310,0,310.0,0.0,captured,2026-11-18 08:45:20.405456+09:00,310.0,0.0,310.0
33,2026-11-17 16:30:00+09:00,0,0,0,0,0.0,0.0,captured,2026-11-18 08:45:20.405456+09:00,0.0,0.0,0.0


**What it shows.** **Observed:**

- **The file states its unit on its first line:** 単位[MW平均], "unit: MW, averaged" over the slot. DWD's MW are the file's numbers unchanged, as they should be for a file already in MW
- **All 48 slots of the 17th agree at every link:** the numbers read by hand, the parser's ODS row, DWD's newest version, and the label's two parts, with the label their sum
- **Every newest version is captured:** the 17 slots up to 08:00 at 08:45 on the 17th, the 17th's own capture, and the 31 from 08:30 at 08:45 on the 18th. The morning's values came again on the 18th unchanged, so they kept their first `known_at`
- Curtailment is zero all day. The day was dull: 1,487 MW at noon

**Learned → design:** nothing to change: the label is the file's.

### 2.2 What T knew of D−2: the persistence feature

**Step.** The persistence baseline predicts each slot of D with the same slot of D−2, 15 November, as known at T (`solar_d2_mw` in `dws_solar_features`). The file captured at 08:45 on the 16th is the newest TEPCO file before T. Its 15 November, read by hand, is compared with the feature at T. Every version of the 15th in DWD shows when each slot became known, since the 15th straddles capture start. The same day in the file captured on the 18th shows whether TEPCO changed any of it after T.

In [6]:
d2 = D - timedelta(days=2)
_, known = tepco_by_hand(KNOWN)
known_d2 = known[known["slot_start"].dt.date == d2][["slot_start", "solar"]]
later_d2 = truth[truth["slot_start"].dt.date == d2][["slot_start", "solar"]]

feature = sql(f"""
SELECT slot_start, solar_d2_mw FROM `{DWS}.dws_solar_features`
WHERE decision_time = {AT_T} ORDER BY slot_start
""")
feature["d2_slot"] = feature["slot_start"].dt.tz_convert("Asia/Tokyo") - timedelta(days=2)
check = known_d2.merge(feature, left_on="slot_start", right_on="d2_slot", suffixes=("", "_d"))
changed = known_d2.merge(later_d2, on="slot_start", suffixes=("_at_t", "_later"))

versions = sql(f"""
SELECT slot_start, basis, known_at, known_until FROM dwd_tepco_actuals
WHERE measure = 'solar' AND DATE(slot_start, 'Asia/Tokyo') = '{d2}'
""")
versions["known_at"] = versions["known_at"].dt.tz_convert("Asia/Tokyo")
print(f"D−2 slots: {len(known_d2)} in the file known at T, {len(feature)} features at T")
print("feature differs from the file known at T:", int((check.solar != check.solar_d2_mw).sum()))
print("slots TEPCO changed after T:", int((changed.solar_at_t != changed.solar_later).sum()))
print("versions of the 15th's solar:", len(versions), "for", versions.slot_start.nunique(), "slots")
(
    versions.assign(slot=versions.slot_start.dt.tz_convert("Asia/Tokyo").dt.strftime("%H:%M"))
    .groupby(["basis", "known_at"])
    .agg(slots=("slot", "count"), first=("slot", "min"), last=("slot", "max"))
)

[1.2 MB scanned, 0 s in BigQuery]


[4.7 MB scanned, 0 s in BigQuery]
D−2 slots: 48 in the file known at T, 48 features at T
feature differs from the file known at T: 0
slots TEPCO changed after T: 0
versions of the 15th's solar: 48 for 48 slots


slots  first   last
basis    known_at                                             
captured 2026-11-15 11:04:16.643393+09:00      1  10:00  10:00
         2026-11-15 11:11:46.678216+09:00      1  10:30  10:30
         2026-11-15 11:36:23.477086+09:00      1  11:00  11:00
         2026-11-15 12:24:01.537490+09:00      1  11:30  11:30
         2026-11-15 14:02:48.862189+09:00      3  12:00  13:00
         2026-11-16 08:45:20.321369+09:00     21  13:30  23:30
rule     2026-11-15 01:30:00.000000+09:00      1  00:00  00:00
         2026-11-15 02:00:00.000000+09:00      1  00:30  00:30
         2026-11-15 02:30:00.000000+09:00      1  01:00  01:00
         2026-11-15 03:00:00.000000+09:00      1  01:30  01:30
         2026-11-15 03:30:00.000000+09:00      1  02:00  02:00
         2026-11-15 04:00:00.000000+09:00      1  02:30  02:30
         2026-11-15 04:30:00.000000+09:00      1  03:00  03:00
         2026-11-15 05:00:00.000000+09:00      1  03:30  03:30
         2026-11-15 05:30:00.000000+09:00      1  04:00  04:00
         2026-11-15 06:00:00.000000+09:00      1  04:30  04:30
         2026-11-15 06:30:00.000000+09:00      1  05:00  05:00
         2026-11-15 07:00:00.000000+09:00      1  05:30  05:30
         2026-11-15 07:30:00.000000+09:00      1  06:00  06:00
         2026-11-15 08:00:00.000000+09:00      1  06:30  06:30
         2026-11-15 08:30:00.000000+09:00      1  07:00  07:00
         2026-11-15 09:00:00.000000+09:00      1  07:30  07:30
         2026-11-15 09:30:00.000000+09:00      1  08:00  08:00
         2026-11-15 10:00:00.000000+09:00      1  08:30  08:30
         2026-11-15 10:30:00.000000+09:00      1  09:00  09:00
         2026-11-15 11:00:00.000000+09:00      1  09:30  09:30

**What it shows.** **Observed:**

- **All 48 slots of the 15th in the file captured before T equal the feature at T,** and TEPCO changed none of them by the 18th
- **The 15th straddles capture start (11:04 that day), and its 48 slots became known in four ways, all before T:**
  - the 20 slots up to 09:30 by the history rule, slot start + 90 minutes, each before capture start;
  - 10:00 to 11:30 one by one, by the test captures of the 15th's late morning;
  - 12:00 to 13:00 by the re-capture at 14:02, a fetch like any other;
  - 13:30 to 23:30 by the 16th's 08:45 capture, 15 minutes before T

**Learned → design:** every way a value becomes known meets in one day, and the feature took each version as known at T. The persistence prediction for D is this feature, so it is checked too. The day lies after the backtest's last fold, October, so it has no outcome row to trace.

### 2.3 The weather at T: one point, from the captured forecast to the feature table

**Step.** The live forecast captured at 08:45 on the 16th holds every hour of the 17th for the nine points, one block per point in the order the capture asked for them. The point with the largest weight at T is followed here, with MSM, the first model in the fill order. Its block is read by hand. Radiation is the "average of the preceding hour" and other variables "an instantaneous value for the indicated hour" (**Source**: [Open-Meteo forecast API](https://open-meteo.com/en/docs)), so both slots of an hour take the radiation stamped at its end, while temperature is interpolated to the slot's middle, a quarter of the way to the next hour for a slot on the hour and three quarters for one on the half hour (`dwd_weather_slots`). The parser's choice of point for each block is checked against the coordinates of each block. The values are compared hour by hour with `dwd_weather_live`, then slot by slot with `dwd_weather_slots` and with `dws_weather` at T.

In [7]:
blocks = json.loads(RAW.get(WEATHER))
names = [name for name, _, _ in WEATHER_POINTS]
ods_points = sql(f"""
SELECT DISTINCT point, grid_latitude, grid_longitude FROM `{ODS}.openmeteo_live`
WHERE source_file = '{WEATHER}'
""").set_index("point")
grid = pd.DataFrame(
    {"latitude": b["latitude"], "longitude": b["longitude"]} for b in blocks
).set_index(pd.Index(names, name="point"))
ods_points = ods_points.reindex(grid.index)
print(
    "blocks:",
    len(blocks),
    "| the parser's point for each block has its coordinates:",
    bool(
        (
            (grid.latitude == ods_points.grid_latitude)
            & (grid.longitude == ods_points.grid_longitude)
        ).all()
    ),
)

weights = sql(f"""
SELECT point, weight FROM `{DWS}.dws_point_weights`
WHERE decision_time = {AT_T} ORDER BY weight DESC
""")
POINT = weights.point.iloc[0]
hourly = pd.DataFrame(blocks[names.index(POINT)]["hourly"])
hourly["time"] = pd.to_datetime(hourly["time"], utc=True).dt.tz_convert("Asia/Tokyo")
hours = hourly.set_index("time")[["shortwave_radiation_jma_msm", "temperature_2m_jma_msm"]]
hours.columns = ["ghi", "temperature"]

slots = pd.date_range(f"{D} 09:00", f"{D} 13:30", freq="30min", tz="Asia/Tokyo")
a = slots.floor("h")
b = a + pd.Timedelta(hours=1)
share = np.where(slots.minute == 0, 0.25, 0.75)
expected = pd.DataFrame(
    {
        "slot_start": slots,
        "ghi_by_hand": hours.ghi.reindex(b).to_numpy(),
        "temperature_by_hand": hours.temperature.reindex(a).to_numpy()
        + share
        * (hours.temperature.reindex(b).to_numpy() - hours.temperature.reindex(a).to_numpy()),
    }
)

stored = sql(f"""
WITH capture AS (
  SELECT ANY_VALUE(captured_at) AS captured_at FROM dwd_weather_live WHERE source_file = '{WEATHER}'
)
SELECT s.slot_start, l.ghi_w_m2 AS live_ghi_at_b,
  s.ghi_w_m2 AS slots_ghi, s.temperature_c AS slots_temperature,
  w.ghi_w_m2 AS dws_ghi, w.temperature_c AS dws_temperature, w.source AS dws_source,
  w.captured_at = c.captured_at AS dws_from_this_capture
FROM dwd_weather_slots AS s
CROSS JOIN capture AS c
JOIN dwd_weather_live AS l
  ON l.point = s.point AND l.model = s.model AND l.captured_at = c.captured_at
  AND l.valid_time = TIMESTAMP_ADD(TIMESTAMP_TRUNC(s.slot_start, HOUR), INTERVAL 1 HOUR)
LEFT JOIN `{DWS}.dws_weather` AS w
  ON w.decision_time = {AT_T} AND w.point = s.point AND w.model = s.model
  AND w.slot_start = s.slot_start
WHERE s.source = 'live' AND s.point = '{POINT}' AND s.model = 'jma_msm'
  AND s.captured_at = c.captured_at
  AND DATE(s.slot_start, 'Asia/Tokyo') = '{D}'
""")
stored["slot_start"] = stored["slot_start"].dt.tz_convert("Asia/Tokyo")
weather = expected.merge(stored, on="slot_start")
print(f"point: {POINT} (weight {weights.weight.iloc[0]:.3f}) | slots compared: {len(weather)}")
print(
    "radiation differs, raw to live to slots to dws:",
    int(
        (
            ~np.isclose(weather.ghi_by_hand, weather.live_ghi_at_b)
            | ~np.isclose(weather.live_ghi_at_b, weather.slots_ghi)
            | ~np.isclose(weather.slots_ghi, weather.dws_ghi)
        ).sum()
    ),
)
print(
    "temperature differs, by hand to slots to dws:",
    int(
        (
            ~np.isclose(weather.temperature_by_hand, weather.slots_temperature)
            | ~np.isclose(weather.slots_temperature, weather.dws_temperature)
        ).sum()
    ),
)
weather

[0.2 MB scanned, 0 s in BigQuery]
blocks: 9 | the parser's point for each block has its coordinates: True


[0.2 MB scanned, 0 s in BigQuery]


[39.1 MB scanned, 0 s in BigQuery]
point: mito (weight 0.235) | slots compared: 10
radiation differs, raw to live to slots to dws: 0
temperature differs, by hand to slots to dws: 0


,slot_start,ghi_by_hand,temperature_by_hand,live_ghi_at_b,slots_ghi,slots_temperature,dws_ghi,dws_temperature,dws_source,dws_from_this_capture
0,2026-11-17 09:00:00+09:00,198.0,11.325,198.0,198.0,11.325,198.0,11.325,live,True
1,2026-11-17 09:30:00+09:00,198.0,11.575,198.0,198.0,11.575,198.0,11.575,live,True
2,2026-11-17 10:00:00+09:00,258.0,11.825,258.0,258.0,11.825,258.0,11.825,live,True
3,2026-11-17 10:30:00+09:00,258.0,12.075,258.0,258.0,12.075,258.0,12.075,live,True
4,2026-11-17 11:00:00+09:00,219.0,12.250,219.0,219.0,12.250,219.0,12.250,live,True
5,2026-11-17 11:30:00+09:00,219.0,12.350,219.0,219.0,12.350,219.0,12.350,live,True
6,2026-11-17 12:00:00+09:00,177.0,12.400,177.0,177.0,12.400,177.0,12.400,live,True
7,2026-11-17 12:30:00+09:00,177.0,12.400,177.0,177.0,12.400,177.0,12.400,live,True
8,2026-11-17 13:00:00+09:00,129.0,12.375,129.0,129.0,12.375,129.0,12.375,live,True
9,2026-11-17 13:30:00+09:00,129.0,12.325,129.0,129.0,12.325,129.0,12.325,live,True


**What it shows.** **Observed:**

- **The nine blocks come in the capture's order,** and the parser gave each the point whose coordinates it carries
- **Mito has the largest weight at T (0.235).** For its MSM forecast in the ten slots from 09:00 to 13:30 of the 17th:
  - radiation is the value stamped at the end of each slot's hour, both slots alike (198 W/m² for 09:00 and 09:30, stamped 10:00), unchanged from the raw file through `dwd_weather_live`, `dwd_weather_slots` and `dws_weather`;
  - temperature, interpolated by hand to each slot's middle, equals `dwd_weather_slots` and `dws_weather`
- **`dws_weather` serves T from this capture, the live source,** as [notebook 04](04_ods_to_dwd.ipynb), section 3.4, set it for target days from the 17th on

**Learned → design:** nothing to change.

### 2.4 Capacity at T

**Step.** METI's figures were checked by hand in [notebook 03](03_solar_capacity.ipynb), which read the raw files with pandas. [Notebook 04](04_ods_to_dwd.ipynb), section 6, reproduced every release's figure in SQL, and `dwd_capacity_area` equals notebook 04 for all 33 releases. So the hand check of capacity at T starts from DWD's releases. The releases known at T and their TEPCO-area totals as known at T come from `dwd_capacity_releases` and `dwd_capacity_area`. The least-squares line through the newest four, by the quarter they describe, is recomputed here with numpy (x in days from D, so the value at D is the intercept), along with the newest figure carried as it is. Both are compared with `dws_capacity` at T. The weights of 2.3 are each point's share of the newest release known at T, recomputed the same way.

In [8]:
def known_at_t(alias: str) -> str:
    """The version of `alias` current at T: what dbt's as_of macro writes."""
    until = f"({alias}.known_until IS NULL OR {AT_T} < {alias}.known_until)"
    return f"({alias}.known_at <= {AT_T} AND {until})"


figures = sql(f"""
SELECT a.ref_date, a.point, CAST(a.solar_mw AS FLOAT64) AS solar_mw
FROM dwd_capacity_area AS a
JOIN dwd_capacity_releases AS r ON r.ref_date = a.ref_date AND {known_at_t("r")}
WHERE {known_at_t("a")}
""")
totals = figures.groupby("ref_date").solar_mw.sum().sort_index().tail(4)
x = np.array([(ref_date - D).days for ref_date in totals.index], dtype=float)
slope, intercept = np.polyfit(x, totals.to_numpy(), 1)
newest = figures[figures.ref_date == totals.index[-1]]
shares = (newest.set_index("point").solar_mw / newest.solar_mw.sum()).rename("share_by_hand")

capacity = sql(f"SELECT * FROM `{DWS}.dws_capacity` WHERE decision_time = {AT_T}").iloc[0]
print(
    "the newest four releases known at T, TEPCO area (MW):",
    {str(k): round(v, 1) for k, v in totals.items()},
)
print(f"line at D: {intercept:,.3f} MW by hand, {capacity.capacity_mw:,.3f} in dws_capacity")
print(f"slope: {slope:.4f} MW a day by hand, {capacity.capacity_slope_mw_per_day:.4f}")
print(f"carried: {totals.iloc[-1]:,.3f} MW by hand, {capacity.capacity_carried_mw:,.3f}")
print("largest weight difference:", float((shares - weights.set_index("point").weight).abs().max()))

[0.0 MB scanned, 0 s in BigQuery]


[0.1 MB scanned, 0 s in BigQuery]
the newest four releases known at T, TEPCO area (MW): {'2025-06-30': 20031.7, '2025-09-30': 20244.2, '2025-12-31': 20405.6, '2026-03-31': 20577.0}
line at D: 21,037.351 MW by hand, 21,037.351 in dws_capacity
slope: 1.9666 MW a day by hand, 1.9666
carried: 20,577.026 MW by hand, 20,577.026
largest weight difference: 4.716251000846938e-10


**What it shows.** **Observed:**

- **The newest four releases known at T** are the ends of June, September and December 2025 and of March 2026: 20,032 to 20,577 MW in the TEPCO area
- **The line through them, recomputed with numpy,** gives 21,037.351 MW at D, rising 1.97 MW a day. The carried figure is 20,577.026 MW. Both equal `dws_capacity` to the printed precision
- **Each point's weight equals its share of the March 2026 release** within 5 × 10⁻¹⁰

**Learned → design:** nothing to change.

### 2.5 Clear sky at T

**Step.** Each slot's clear-sky output at T is capacity at T × the weighted clear-sky radiation at the nine points ÷ 1000 W/m² (plan, Decisions). It is recomputed here in pandas, from `dim_sun_slots` (the radiation per point and slot, checked against SPA in [notebook 05](05_clear_sky.ipynb)), the weights at T and `capacity_mw` at T. It is compared with `clear_sky_mw` in `dws_solar_features` for every slot of D.

In [9]:
sun = sql(f"""
SELECT point, slot_start, clear_sky_ghi FROM `{DIM}.dim_sun_slots`
WHERE DATE(slot_start, 'Asia/Tokyo') = '{D}'
""")
sun = sun.merge(weights, on="point")
weighted = (sun.weight * sun.clear_sky_ghi).groupby(sun.slot_start).sum()
sky_by_hand = capacity.capacity_mw * weighted / 1000
features = sql(f"""
SELECT slot_start, clear_sky_mw, capacity_mw FROM `{DWS}.dws_solar_features`
WHERE decision_time = {AT_T}
""").set_index("slot_start")
gap = (sky_by_hand - features.clear_sky_mw).abs()
print(
    f"slots: {len(features)} | largest difference: {gap.max():.2e} MW |",
    "capacity in every slot is dws_capacity's:",
    bool((features.capacity_mw == capacity.capacity_mw).all()),
)
shown = pd.DataFrame({"by_hand": sky_by_hand, "dws": features.clear_sky_mw})
shown.index = shown.index.tz_convert("Asia/Tokyo")
shown[shown.by_hand > 0].iloc[::4].round(3)

[0.3 MB scanned, 0 s in BigQuery]


[1.5 MB scanned, 0 s in BigQuery]
slots: 48 | largest difference: 4.16e-08 MW | capacity in every slot is dws_capacity's: True


,by_hand,dws
slot_start,,
2026-11-17 06:00:00+09:00,1.914,1.914
2026-11-17 08:00:00+09:00,6237.346,6237.346
2026-11-17 10:00:00+09:00,11113.370,11113.370
2026-11-17 12:00:00+09:00,11526.544,11526.544
2026-11-17 14:00:00+09:00,7357.480,7357.480
2026-11-17 16:00:00+09:00,326.516,326.516


**What it shows.** **Observed:** clear sky recomputed in pandas equals `clear_sky_mw` in all 48 slots, within 4 × 10⁻⁸ MW. That is the difference of float sums added in another order; the model adds in a fixed order (plan, Decisions). Every slot carries `dws_capacity`'s figure. At noon clear sky is 11,527 MW, and the day delivered 1,487 MW, 13% of it.

**Learned → design:** nothing to change.

## 3. How exact each decision time is

**Step.** A replay at T is flagged by its *fidelity*: *"`exact`: `T` is after capture start, so every input version was captured with its real fetch time"*, and `approximate` otherwise (**Source**: [PRD](../docs/prd.md), Time machine, Fidelity). Section 2.2 found a T after the actuals' capture start that still read 20 rule-dated slots, so "after capture start" and "every input version captured" are not the same. A version dated by the rule is history in two ways: its time comes from the rule, and its value is today's, with any later correction built in. So this section measures, for every decision time, what each input actually read, as the models read it:

| Input | What T reads | Captured when |
| --- | --- | --- |
| Actuals | The versions of D−2's 48 slots current at T, which persistence reads; and D−8 to D−2, the seven complete days the clear-sky × ratio baseline of step 7 reads | Every version read has `basis = 'captured'` |
| Weather | The source `dws_weather` serves for D: the live capture, lead 2 of Previous Runs, or a Single Run | The live capture: the history-only sources have no capture start (section 1.1) |
| Capacity | The four releases `dws_capacity`'s line goes through, as known at T | Every release read has `basis = 'captured'` |

In [ ]:
import numpy as np

DWS = f"{PROJECT}.{setting('JTF_DBT_DATASET_PREFIX')}_dws"
# What dbt's as_of macro writes, for an alias {v} and the decision time d.decision_time
AS_OF = (
    "{v}.known_at <= d.decision_time"
    " AND ({v}.known_until IS NULL OR d.decision_time < {v}.known_until)"
)

actuals = sql(f"""
WITH decisions AS (SELECT DISTINCT decision_time, target_day FROM `{DWS}.dws_decision_slots`),
days AS (
  SELECT decision_time, target_day, day
  FROM decisions, UNNEST(GENERATE_DATE_ARRAY(DATE_SUB(target_day, INTERVAL 8 DAY),
                                             DATE_SUB(target_day, INTERVAL 2 DAY))) AS day
)
SELECT d.decision_time,
  COUNTIF(d.day = DATE_SUB(d.target_day, INTERVAL 2 DAY)) AS d2_slots,
  COUNTIF(d.day = DATE_SUB(d.target_day, INTERVAL 2 DAY) AND v.basis = 'rule') AS d2_rule,
  COUNT(*) AS week_slots, COUNTIF(v.basis = 'rule') AS week_rule
FROM days AS d
JOIN dwd_tepco_actuals AS v
  ON v.measure = 'solar' AND DATE(v.slot_start, 'Asia/Tokyo') = d.day AND {AS_OF.format(v="v")}
GROUP BY 1
""")
weather = sql(f"""
SELECT decision_time, STRING_AGG(DISTINCT source ORDER BY source) AS weather_source
FROM `{DWS}.dws_weather` GROUP BY 1
""")
capacity = sql(f"""
SELECT d.decision_time, d.releases, COUNTIF(r.basis = 'rule') AS rule_releases,
  TIMESTAMP_DIFF(d.decision_time, d.newest_known_at, DAY) AS newest_release_age_days
FROM `{DWS}.dws_capacity` AS d
JOIN dwd_capacity_releases AS r
  ON r.ref_date BETWEEN d.oldest_ref_date AND d.newest_ref_date AND {AS_OF.format(v="r")}
GROUP BY 1, 2, 4
""")
reads = actuals.merge(weather, on="decision_time").merge(capacity, on="decision_time")
reads["decision_time"] = reads["decision_time"].dt.tz_convert("Asia/Tokyo")
reads = reads.sort_values("decision_time").reset_index(drop=True)
print(
    f"decision times: {len(reads)} | every one read 48 slots of D−2: {(reads.d2_slots == 48).all()}"
    f" | four releases: {(reads.releases == 4).all()}"
)

[5.5 MB scanned, 0 s in BigQuery]


[30.0 MB scanned, 0 s in BigQuery]


[0.0 MB scanned, 0 s in BigQuery]
decision times: 1007 | every one read 48 slots of D−2: True | four releases: True


**What it shows.** Every decision time read all 48 slots of D−2 and four releases, as the models do.

**3.1** Each input, exact or approximate, at every decision time: the counts over the whole range, then each decision time around capture start.

In [ ]:
exact = pd.DataFrame(
    {
        "decision_time": reads.decision_time,
        "actuals_d2": np.where(reads.d2_rule == 0, "exact", "approximate"),
        "actuals_week": np.where(reads.week_rule == 0, "exact", "approximate"),
        "weather": np.where(reads.weather_source == "live", "exact", "approximate"),
        "capacity": np.where(reads.rule_releases == 0, "exact", "approximate"),
    }
)
print(exact.drop(columns="decision_time").apply(pd.Series.value_counts).fillna(0).astype(int))
around = reads.decision_time >= pd.Timestamp("2026-11-12", tz="Asia/Tokyo")
reads.loc[
    around,
    [
        "decision_time",
        "d2_rule",
        "week_rule",
        "weather_source",
        "rule_releases",
        "newest_release_age_days",
    ],
]

             actuals_d2  actuals_week  weather  capacity
approximate        1005          1007     1004      1007
exact                 2             0        3         0


,decision_time,d2_rule,week_rule,weather_source,rule_releases,newest_release_age_days
1000,2026-11-12 09:00:00+09:00,48,336,previous_runs,4,85
1001,2026-11-13 09:00:00+09:00,48,336,previous_runs,4,86
1002,2026-11-14 09:00:00+09:00,48,336,previous_runs,4,87
1003,2026-11-15 09:00:00+09:00,48,336,single_runs,4,88
1004,2026-11-16 09:00:00+09:00,20,308,live,4,89
1005,2026-11-17 09:00:00+09:00,0,260,live,4,90
1006,2026-11-18 09:00:00+09:00,0,212,live,4,91


**What it shows.** **Observed:**

- **Actuals, D−2: exact at 2 of 1,007 decision times,** 09:00 on the 17th and 18th. At 09:00 on the 16th, after the actuals' capture start, 20 of the 48 slots read were still rule-dated (section 2.2)
- **Actuals, the seven complete days: exact at none yet.** At 09:00 on the 18th, 212 of 336 slots are rule-dated: the week reaches back before capture start, until 09:00 on the 23rd, whose D−8 is the 16th
- **Weather: exact at 3, from 09:00 on the 16th,** when `dws_weather` serves the live capture. Before that it serves Previous Runs, and a Single Run on the 15th ([notebook 04](04_ods_to_dwd.ipynb), section 3.4)
- **Capacity: exact at none.** All four releases read are rule-dated at every decision time, and will be until four captured releases exist, about a year after the first (due by mid-December)

**Learned → design:** fidelity is not one date. It is per input, from the `basis` of the versions read, and a model is exact at T only where every input it reads is: persistence from the 17th, a model reading the seven days from the 23rd. Taken literally, capacity would keep every model approximate for a year; section 3.3 asks whether its rule-dated releases deserve that.

**3.2** The capacity releases a decision time reads are all rule-dated until captured ones replace them. How long before T the newest of them became known, over every decision time.

In [ ]:
ages = reads.newest_release_age_days
print(
    f"the newest release read was known {ages.min()} to {ages.max()} days before T,",
    f"median {ages.median():.0f}",
)
print("decision times reading a release known 16 days or less before T:", int((ages <= 16).sum()))

the newest release read was known 0 to 125 days before T, median 46
decision times reading a release known 16 days or less before T: 178


**What it shows.** **Observed:** the newest release read was known 0 to 125 days before T, median 46, and 178 decision times read a release known within 16 days. A rule-dated release is often recent at T, so its date matters, not only its value.

**3.3** Whether a rule-dated release is history in the way rule-dated actuals are. Per rule, how many releases, whether any release's files changed after its `known_at` (their newest `Last-Modified` later than it), and how long after the files went up each was dated known.

In [ ]:
sql("""
SELECT basis, rule, COUNT(*) AS releases,
  COUNTIF(GREATEST(a_last_modified, b_last_modified) > known_at) AS files_changed_after_known,
  MIN(TIMESTAMP_DIFF(known_at, GREATEST(a_last_modified, b_last_modified), DAY))
    AS min_days_after_files,
  MAX(TIMESTAMP_DIFF(known_at, GREATEST(a_last_modified, b_last_modified), DAY))
    AS max_days_after_files
FROM dwd_capacity_releases GROUP BY 1, 2 ORDER BY 3 DESC
""")

[0.0 MB scanned, 0 s in BigQuery]


,basis,rule,releases,files_changed_after_known,min_days_after_files,max_days_after_files
0,rule,announced,23,0,0,16
1,rule,last_modified,9,0,0,0
2,rule,margin,1,0,16,16


**What it shows.** **Observed:**

- **No release's files changed after its `known_at`:** the figures read are the ones published
- **The rule never dates a release before its files went up:**
  - 23 releases at the end of their announcement day, up to 16 days after the files;
  - 9 at the files' upload, which came after the announcement;
  - 1, never announced, 16 days after its files (end of June 2017, known only from 2024)

**Learned → design:** a rule-dated capacity release is not history the way rule-dated actuals are. Its value is the published one, and its date is never early: at worst an announced release is read a day late. A capture has the same precision, since it learns of a release at the next morning's fetch. So capacity's rule-dated releases count as exact. What stays approximate is the actuals and the weather before their capture: their values carry TEPCO's later corrections, and their times come from the models' assumed delays.

## The dbt tests this leads to

| Test | Check | On |
| --- | --- | --- |
| `<model>__rule_only_before_capture_start` (generic `rule_before_capture_start`) | C | The actuals, the capacity releases and their figures (`dwd_capacity_area`) |
| `<model>__rule_never_after_its_first_fetch` | D | The actuals, the capacity releases and figures, Previous Runs, Single Runs |
| `<model>__captured_never_before_its_fetch` | A | The capacity figures and Single Runs. The actuals and releases already test B, and Previous Runs tests A on its corrections |
| `<model>__no_known_at_after_newest_fetch` (generic `known_at_within_fetches`) | E | All seven |

The two generic tests read the ODS manifest ([tests/generic](../warehouse/dbt/tests/generic/)); the other two are expressions on the model's own fetch column. 17 tests in all, each seen to fail: the actuals built with their rule applied past capture start fail all three of theirs, and a violation planted in a copy of each other model's rows fails its test.

## What this exploration made known

| We assumed | We found | So |
| --- | --- | --- |
| The plan's three checks after every load apply to every source | Previous Runs and Single Runs came only by `initial` fetches: no daily capture, so no capture start for "rule before capture start" to compare against (section 1.1) | A rule is also checked against the fetch that first held each version: never dated after it. Every source gets that check; "before capture start" stays for the sources with a daily capture |
| A captured version carries the fetch time of a file holding the value (plan) | True, exactly, of every captured version today (section 1.2). The weather dates a correction no earlier than the version it replaces, so one could come later than its fetch | Exactly at its fetch where the model says so (the actuals, the capacity releases); elsewhere, never before it |
| `known_at` is right per source, as notebook 04 designed it | No version among 10.1 M in seven models breaks a check that applies to it (section 1.2) | The four checks become data tests after every load, 17 over the seven models |
| On a day served from captures, the warehouse is true to the raw files | Target day 17 November, traced by hand. The truth: all 48 slots equal from the raw file through ODS and DWD to the label. D−2: all 48 slots equal to the feature at T, and unchanged after T. The weather at T (Mito, MSM, ten slots): equal from the raw forecast through every layer, radiation and the interpolated temperature both. Capacity at T: the line, slope, carried figure and weights equal `dws_capacity` when recomputed with numpy. Clear sky: within 4 × 10⁻⁸ MW in every slot (section 2) | Step 5's check "one known day checked by hand against the source files" holds |
| TEPCO's actuals are MW, labelled by slot start (plan; notebook 05) | The file's first line says 単位[MW平均], "unit: MW, averaged", and DWD's MW are its numbers unchanged (section 2.1) | Confirmed |
| A day near capture start mixes the ways a value becomes known | The 15th's 48 slots were known four ways before T: 20 by the rule, 4 by the 15th's test captures, 3 by a re-capture, 21 by the 16th's capture (section 2.2) | Every way of knowing a value meets in one day, and the feature at T took each version as known |
| A decision time is exact when `T` is after capture start (PRD, Fidelity) | At 09:00 on the 16th, after the actuals' capture start, D−2 still read 20 rule-dated slots; the weather becomes exact a day earlier, the seven days of actuals not until the 23rd (section 3.1) | Fidelity per input, from the `basis` of the versions each decision time read. A run is exact at T only where every input its model reads is |
| A rule-dated version is approximate, whatever the source | Capacity's rule-dated releases hold the published figures (no file changed after its `known_at`), and the rule never dates one early (section 3.3) | Capacity's rule-dated releases count as exact; rule-dated actuals and weather stay approximate |
| (No expectation) | Persistence is exact from 09:00 on the 17th, a model reading the seven days of actuals from the 23rd; the range grows by a day each day (section 3.1) | Every result states its fidelity per input |

### Still open

- **A day of the history era**, before capture start (the actuals and the weather all rule-dated, the weather from Previous Runs), is not traced by hand here: its rules are [notebook 04](04_ods_to_dwd.ipynb)'s, checked on every version in section 1
- **The day's outcome:** it lies after the backtest's last fold, so the prediction is checked as the feature it is (section 2.2), not as an outcome row
- **How approximate the rule-dated actuals and weather are:** the actuals' monthly re-capture on 1 December shows how large and late TEPCO's corrections are, and re-fetched Single Runs in early December show the models' real delays (progress, *Due on a date*)
- **The captured side of capacity and Single Runs has nothing to check yet.** It will once the first captured capacity release arrives (due by mid-December, progress, *Due on a date*) and once a run changes between fetches

## Where the sources and the data disagree

| Where | Says | We found | Handled by |
| --- | --- | --- | --- |
| [Phase-1 plan](../docs/phase1/README.md#ingestion-and-warehouse), Warehouse tests | After every load, `basis = rule` rows sit before their source's capture start | The weather history has no capture start (section 1.1) | The plan now also names the rule's check against the first fetch, for every source |
| The same paragraph | `basis = captured` rows carry the fetch time of a file holding the value | The weather dates a correction no earlier than the version it replaces, which can be later than its fetch (section 1.2) | Never before its fetch; exactly at it where the model says so |
| [PRD](../docs/prd.md), Time machine, Fidelity | *"`exact`: `T` is after capture start, so every input version was captured with its real fetch time"* | After capture start, D−2 still read rule-dated slots; and capacity's rule-dated releases are as precise as a capture (section 3) | Fidelity per input, from what each decision time read; the PRD and the plan updated (2026-11-18) |


## Sources

| Page | Supports |
| --- | --- |
| [Notebook 04](04_ods_to_dwd.ipynb), sections 1, 2.2, 3.1, 3.2 and 6 | Capture start per source, the late backfill, the weather's rules, capacity's rule |
| [weather.sql](../warehouse/dbt/macros/weather.sql), `weather_versions` | The weather history's versions: the first by rule, a later one at its fetch, never before the version it replaces |
| [ingest README](../jtf/src/jtf/ingest/README.md) | The history-only sources have no daily schedule |
| [TEPCO area supply and demand](https://www.tepco.co.jp/forecast/html/area_jukyu-j.html) | The monthly actuals file traced in section 2: MW averaged over each slot, labelled by its start |
| [Open-Meteo forecast API](https://open-meteo.com/en/docs) | Radiation as the average of the preceding hour; other variables instantaneous |
| [Notebook 03](03_solar_capacity.ipynb) | METI's capacity figures read by hand from the raw files |
| [Phase-1 plan](../docs/phase1/README.md#ingestion-and-warehouse), Warehouse tests | The checks after every load |
| [PRD](../docs/prd.md), Time machine, Fidelity | The definition of `exact` and `approximate` |
